# Challenge 1: Structured Agents with FoundryChatClient

## Why Structured Outputs Matter

Most agent tutorials have agents return **free text** — then you regex-parse it and pray.
MAF agents return **Pydantic models** — typed, validated, machine-readable data that can be
routed deterministically through a workflow graph.

In this challenge you'll build agents that:
- Use `FoundryChatClient` (lightweight, local agents — no server-side resource creation)
- Return **Pydantic structured outputs** via `response_format`
- Use MAF's `@tool` decorator with type annotations
- Produce typed data that downstream agents and workflow executors can consume reliably

| Agent | Structured Output | Tools | Purpose |
|-------|------------------|-------|---------|
| **Triage** | `TriageResult` | `check_alert_history`, `get_runbook` | Classify severity, decide response path |
| **Diagnostics** | `DiagnosticsResult` | `get_metrics`, `get_logs`, `check_dependencies` | Find root cause with evidence |
| **Remediation Planner** | `RemediationPlan` | *(none — plans only)* | Determine fix strategy from diagnostics |
| **Verification** | `VerificationResult` | `get_health_status`, `run_smoke_test` | Confirm fix worked |

---

## How This Challenge Works

1. The **Triage Agent** is provided as a complete reference
2. You define the **Pydantic models** for remaining agents' outputs
3. You build the **Diagnostics**, **Remediation Planner**, and **Verification** agents
4. Each agent has a validation cell that asserts on the structured output

> **Key Insight**: When agents return typed data (`TriageResult.severity == "critical"`),
> you can route workflows with Python conditionals — not fragile string matching.

## Setup

In [ ]:
import os
import sys
import json
from typing import Literal

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient, OpenAIChatOptions

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    get_health_status, run_smoke_test,
)

load_dotenv("../.env")

# Create client — works with GitHub Models OR Azure AI Foundry
if os.environ.get("GITHUB_TOKEN"):
    client = OpenAIChatClient(
        model=os.environ.get("MODEL_NAME", "gpt-4o"),
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.inference.ai.azure.com",
    )
    print("✅ Connected to GitHub Models")
elif os.environ.get("FOUNDRY_PROJECT_ENDPOINT"):
    from agent_framework.foundry import FoundryChatClient
    from azure.identity import AzureCliCredential
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ.get("FOUNDRY_MODEL", "gpt-4o"),
        credential=AzureCliCredential(),
    )
    print("✅ Connected to Azure AI Foundry")
else:
    raise ValueError("Set GITHUB_TOKEN or FOUNDRY_PROJECT_ENDPOINT in .env")

# Load incident data
with open("../data/incidents.json") as f:
    incidents = json.load(f)

incident = incidents[0]  # Critical: payment-api OOM
print(f"\n🚨 Incident: {incident['title']}")
print(f"   Service: {incident['service']} | Severity: {incident['severity']}")
print(f"   Type: {incident['incident_type']}")
print(f"   {incident['description']}")


c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


🚨 Incident: Payment API P99 Latency > 30s
   Service: payment-api | Severity: critical
   Type: high_latency
   payment-api pod-3 OOMKilled. P99 latency spiked to 32s (baseline: 200ms). Connection pool exhausted with 312 requests queued.


---
## Step 1: Define Structured Output Models

These Pydantic models define the **contract** between agents.
When you set `response_format=TriageResult`, MAF forces the LLM to return
valid JSON matching this schema — no parsing errors, no hallucinated fields.

### TriageResult (provided)

In [2]:
class TriageResult(BaseModel):
    """Structured output from the Triage Agent."""
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool = Field(description="Whether this alert pattern has been seen before")
    auto_remediation_allowed: bool = Field(description="Whether the runbook permits auto-fix")
    root_cause_hypothesis: str = Field(description="Initial hypothesis based on alert history")
    recommended_action: str = Field(description="What the Diagnostics Agent should investigate")
    escalation_threshold_minutes: int = Field(description="Minutes before human escalation")

print("\u2705 TriageResult model defined")
print(f"   Fields: {list(TriageResult.model_fields.keys())}")

✅ TriageResult model defined
   Fields: ['severity', 'is_recurring', 'auto_remediation_allowed', 'root_cause_hypothesis', 'recommended_action', 'escalation_threshold_minutes']


---

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Define `DiagnosticsResult`

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>


In [ ]:
# Create a Pydantic BaseModel called DiagnosticsResult with these fields:
# - root_cause: str — what's actually broken
# - evidence: list[str] — metrics/logs/data points that prove it
# - affected_components: list[str] — pods, services, or components impacted
# - confidence: float — confidence score between 0.0 and 1.0 (use ge=0.0, le=1.0)
# - recommended_fix: str — specific remediation action with target details
# - requires_restart: bool — whether the fix requires restarting a pod or service
# Use Field(description=...) for each field, like the TriageResult above.


✅ DiagnosticsResult model defined
   Fields: ['root_cause', 'evidence', 'affected_components', 'confidence', 'recommended_fix', 'requires_restart']


---

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Define `RemediationPlan`

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>


In [ ]:
# Create a Pydantic BaseModel called RemediationPlan with these fields:
# - action: Literal["restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate"]
# - target_service: str — service to act on
# - target_details: str — specific pod ID, replica count, flag name, or cache name
# - risk_level: Literal["low", "medium", "high"]
# - estimated_downtime_seconds: int (ge=0) — estimated downtime in seconds
# - rollback_strategy: str — how to undo this action if it fails
# - requires_approval: bool — whether human approval is needed before execution
# Use Field(description=...) for each field. Follow the same pattern as TriageResult.


✅ RemediationPlan model defined
   Fields: ['action', 'target_service', 'target_details', 'risk_level', 'estimated_downtime_seconds', 'rollback_strategy', 'requires_approval']


---

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Define `VerificationResult`

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the code from the comments.

</div>


In [ ]:
# Create a Pydantic BaseModel called VerificationResult with these fields:
# - service_healthy: bool — whether the service is healthy after remediation
# - tests_passed: int (ge=0) — number of smoke tests that passed
# - tests_failed: int (ge=0) — number of smoke tests that failed
# - verification_status: Literal["pass", "fail", "degraded"] — overall outcome
# - details: str — summary of health check and test results
# Use Field(description=...) for each field.


✅ VerificationResult model defined
   Fields: ['service_healthy', 'tests_passed', 'tests_failed', 'verification_status', 'details']


---
## Step 2: Build the Triage Agent (REFERENCE)

Study this carefully — it demonstrates the pattern:
1. `client` is already created in the setup cell above (GitHub Models or Foundry)
2. `response_format=TriageResult` forces structured JSON output
3. Tools passed directly to the `Agent` constructor
4. `agent.run()` returns an `AgentResponse` whose `.text` is valid JSON
5. Validate with `TriageResult.model_validate_json(response.text)`


In [ ]:
# client is already created in setup cell above — reuse it here

triage_agent = Agent(
    client,
    id="triage-agent",
    name="TriageAgent",
    instructions=(
        "You are an incident Triage Agent — the first responder when a production alert fires.\n"
        "\n"
        "When given an alert, you MUST:\n"
        "1. Call check_alert_history to see if this is a recurring pattern\n"
        "2. Call get_runbook with the incident_type to find the operations playbook\n"
        "\n"
        "Based on the results, assess:\n"
        "- Severity (critical/high/medium/low) considering recurrence and blast radius\n"
        "- Whether auto-remediation is safe (from the runbook)\n"
        "- Initial root cause hypothesis\n"
        "- What the Diagnostics Agent should investigate next\n"
        "\n"
        "CONSTRAINTS:\n"
        "- Do NOT attempt to fix anything — only classify and route\n"
        "- Always use both tools before making your assessment\n"
        "- Be specific about what to investigate next"
    ),
    tools=[check_alert_history, get_runbook],
    default_options=OpenAIChatOptions(response_format=TriageResult),
)

print("✅ Triage Agent created")


✅ Triage Agent created


### Run the Triage Agent

In [7]:
# Run the triage agent and validate structured output
triage_response = await triage_agent.run(
    f"New alert fired:\n"
    f"Title: {incident['title']}\n"
    f"Service: {incident['service']}\n"
    f"Type: {incident['incident_type']}\n"
    f"Description: {incident['description']}"
)

# Parse and validate the structured output
triage_result = TriageResult.model_validate_json(triage_response.text)

print("\U0001f3af TRIAGE RESULT (structured):")
print(f"   severity: {triage_result.severity}")
print(f"   is_recurring: {triage_result.is_recurring}")
print(f"   auto_remediation_allowed: {triage_result.auto_remediation_allowed}")
print(f"   root_cause_hypothesis: {triage_result.root_cause_hypothesis}")
print(f"   recommended_action: {triage_result.recommended_action}")
print(f"   escalation_threshold_minutes: {triage_result.escalation_threshold_minutes}")

# THIS is why structured outputs matter — deterministic routing:
print(f"\n\u27a1\ufe0f  Routing decision: severity=={triage_result.severity!r} \u2192 ", end="")
if triage_result.severity == "critical":
    print("FULL PIPELINE (diagnose \u2192 remediate \u2192 verify \u2192 comms)")
elif triage_result.severity == "high":
    print("EXPEDITED PIPELINE (diagnose \u2192 remediate \u2192 verify)")
else:
    print("MONITOR ONLY (log and notify)")

🎯 TRIAGE RESULT (structured):
   severity: high
   is_recurring: True
   auto_remediation_allowed: True
   root_cause_hypothesis: Memory leak in the payment-api connection pool, triggered periodically after batch job execution.
   recommended_action: The Diagnostics Agent should inspect memory metrics of payment-api pod-3, verify the presence of connection pool leaks, and ensure a pod restart addresses the issue effectively.
   escalation_threshold_minutes: 15

➡️  Routing decision: severity=='high' → EXPEDITED PIPELINE (diagnose → remediate → verify)


In [8]:
# ✅ Validation
assert isinstance(triage_result, TriageResult), "Output must be a TriageResult"
assert triage_result.severity in ("critical", "high", "medium", "low")
assert isinstance(triage_result.is_recurring, bool)
assert isinstance(triage_result.auto_remediation_allowed, bool)
assert len(triage_result.root_cause_hypothesis) > 10
assert triage_result.escalation_threshold_minutes > 0
print("✅ Triage Agent validation passed — structured output is correct")

✅ Triage Agent validation passed — structured output is correct


---
## Step 3: Build the Diagnostics Agent

Study the **Triage Agent** above — notice the pattern:
1. `Agent(client, ...)` with the shared `FoundryChatClient`
2. `instructions=` telling the agent WHAT to do and HOW
3. `tools=[...]` giving it access to infrastructure APIs
4. `default_options=OpenAIChatOptions(response_format=...)` for structured output

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Create `diagnostics_agent`

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the agent from the comments.

</div>


In [ ]:
# Create a MAF Agent called diagnostics_agent using the existing `client` variable.
# - id="diagnostics-agent", name="DiagnosticsAgent"
# - tools: [get_metrics, get_logs, check_dependencies] (already imported)
# - default_options: OpenAIChatOptions(response_format=DiagnosticsResult)
# - instructions should tell the agent to:
#   1. Call get_metrics with the service name for memory/latency/error_rate
#   2. Call get_logs with severity='ERROR' to find error patterns
#   3. Call check_dependencies to identify cascading failures
#   4. Synthesize findings: confidence 0.9+ only if all 3 tools agree
#   5. Never guess without tool data
# Follow the exact same pattern as triage_agent above.

✅ Diagnostics Agent created


### Run & Validate the Diagnostics Agent

In [10]:
# Feed it the triage result — this is how agents chain: typed output → typed input
diag_response = await diagnostics_agent.run(
    f"Investigate this incident based on triage assessment:\n"
    f"Service: {incident['service']}\n"
    f"Triage hypothesis: {triage_result.root_cause_hypothesis}\n"
    f"Recommended investigation: {triage_result.recommended_action}\n"
    f"Original alert: {incident['description']}"
)

diag_result = DiagnosticsResult.model_validate_json(diag_response.text)

print("\U0001f50d DIAGNOSTICS RESULT (structured):")
print(f"   root_cause: {diag_result.root_cause}")
print(f"   confidence: {diag_result.confidence}")
print(f"   affected_components: {diag_result.affected_components}")
print(f"   requires_restart: {diag_result.requires_restart}")
print(f"   evidence:")
for e in diag_result.evidence:
    print(f"     - {e}")
print(f"   recommended_fix: {diag_result.recommended_fix}")

🔍 DIAGNOSTICS RESULT (structured):
   root_cause: The payment-api pod-3 experienced a memory leak, resulting in Out-of-Memory (OOM) errors. This caused connection pool exhaustion as the batch job executed, overwhelming the service.
   confidence: 0.9
   affected_components: ['payment-api-pod-3', 'order-service']
   requires_restart: True
   evidence:
     - Memory utilization for payment-api-pod-3 reached 846 MB out of 1024 MB, triggering an OOMKill.
     - Error logs from payment-api-pod-3 indicated 'java.lang.OutOfMemoryError: Java heap space.'
     - Dependency health showed 'order-service' degrading with high retry volumes, potentially contributing to increased requests to payment-api.
   recommended_fix: Patch the connection pooling logic to address memory leaks and monitor for stability. Immediately restart processes using 'kubectl restart pod payment-api-pod-3'.


In [11]:
# ✅ Validate diagnostics output
assert isinstance(diag_result, DiagnosticsResult), "Output must be DiagnosticsResult"
assert len(diag_result.root_cause) > 10, "Root cause should be descriptive"
assert len(diag_result.evidence) >= 2, "Should have at least 2 pieces of evidence"
assert 0.0 <= diag_result.confidence <= 1.0, "Confidence must be between 0 and 1"
assert len(diag_result.affected_components) >= 1, "Should identify affected components"
assert isinstance(diag_result.requires_restart, bool)
print("✅ Diagnostics Agent validation passed")

✅ Diagnostics Agent validation passed


---
## Step 4: Build the Remediation Planner

This agent **plans** the fix but does NOT execute it (HITL approval comes in Challenge 3).
Key difference: this agent has **NO tools** — it only reasons about what action to take.

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Create `planner`

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the agent from the comments.

</div>


In [ ]:
# Create a MAF Agent called planner using the existing `client` variable.
# - id="remediation-planner", name="RemediationPlanner"
# - NO tools (it only plans, does not execute)
# - default_options: OpenAIChatOptions(response_format=RemediationPlan)
# - instructions should tell the agent to:
#   1. Choose action based on root cause: OOM→restart_pod, high CPU→scale_service,
#      stale cache→flush_cache, dependency down→toggle_feature_flag, unclear→escalate
#   2. Set requires_approval=True for any action affecting production traffic
#   3. Always include a realistic rollback_strategy
#   4. Estimate realistic downtime (pod restart ≈ 30s, scale ≈ 45s)
#   5. risk_level='high' if action could cause brief outage
# Follow the same Agent(...) pattern as triage_agent.


✅ Remediation Planner created


### Run & Validate the Remediation Planner

In [13]:
plan_response = await planner.run(
    f"Create a remediation plan based on this diagnosis:\n"
    f"Service: {incident['service']}\n"
    f"Root cause: {diag_result.root_cause}\n"
    f"Confidence: {diag_result.confidence}\n"
    f"Affected components: {diag_result.affected_components}\n"
    f"Requires restart: {diag_result.requires_restart}\n"
    f"Recommended fix: {diag_result.recommended_fix}"
)

plan_result = RemediationPlan.model_validate_json(plan_response.text)

print("\U0001f6e0\ufe0f REMEDIATION PLAN (structured):")
print(f"   action: {plan_result.action}")
print(f"   target_service: {plan_result.target_service}")
print(f"   target_details: {plan_result.target_details}")
print(f"   risk_level: {plan_result.risk_level}")
print(f"   estimated_downtime_seconds: {plan_result.estimated_downtime_seconds}")
print(f"   rollback_strategy: {plan_result.rollback_strategy}")
print(f"   requires_approval: {plan_result.requires_approval}")

🛠️ REMEDIATION PLAN (structured):
   action: restart_pod
   target_service: payment-api
   target_details: pod-3
   risk_level: high
   estimated_downtime_seconds: 30
   rollback_strategy: Revert the recent deployment on payment-api service and reattempt patching.
   requires_approval: True


In [14]:
# ✅ Validate remediation plan
assert isinstance(plan_result, RemediationPlan)
assert plan_result.action in ("restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate")
assert plan_result.risk_level in ("low", "medium", "high")
assert plan_result.estimated_downtime_seconds >= 0
assert len(plan_result.rollback_strategy) > 5, "Rollback strategy must be specified"
assert isinstance(plan_result.requires_approval, bool)
print("✅ Remediation Planner validation passed")

✅ Remediation Planner validation passed


---
## Step 5: Build the Verification Agent

This agent checks if the fix actually worked by probing the service.

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Create `verifier`

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the agent from the comments.

</div>


In [ ]:
# Create a MAF Agent called verifier using the existing `client` variable.
# - id="verification-agent", name="VerificationAgent"
# - tools: [get_health_status, run_smoke_test] (already imported)
# - default_options: OpenAIChatOptions(response_format=VerificationResult)
# - instructions should tell the agent to:
#   1. Call get_health_status to check the service health endpoints
#   2. Call run_smoke_test to run functional tests
#   3. Set verification_status='pass' only if BOTH health check AND all tests pass
#   4. Set 'degraded' if health OK but some tests fail
#   5. Set 'fail' if health check itself fails
#   6. Count tests_passed and tests_failed accurately from results
# Follow the same Agent(...) pattern as triage_agent.


✅ Verification Agent created


### Run & Validate

In [16]:
verify_response = await verifier.run(
    f"Verify that the remediation was successful:\n"
    f"Service: {incident['service']}\n"
    f"Action taken: {plan_result.action} on {plan_result.target_details}\n"
    f"Expected outcome: Service healthy, latency back to baseline"
)

verify_result = VerificationResult.model_validate_json(verify_response.text)

print("✅ VERIFICATION RESULT (structured):")
print(f"   service_healthy: {verify_result.service_healthy}")
print(f"   tests_passed: {verify_result.tests_passed}")
print(f"   tests_failed: {verify_result.tests_failed}")
print(f"   verification_status: {verify_result.verification_status}")
print(f"   details: {verify_result.details}")

✅ VERIFICATION RESULT (structured):
   service_healthy: True
   tests_passed: 12
   tests_failed: 0
   verification_status: pass
   details: Both health checks and critical path smoke tests for payment-api passed without any issues. Service response time is within expected baseline at 12ms.


In [17]:
assert isinstance(verify_result, VerificationResult)
assert verify_result.verification_status in ("pass", "fail", "degraded")
assert verify_result.tests_passed >= 0
assert isinstance(verify_result.service_healthy, bool)
print("✅ Verification Agent validation passed")

✅ Verification Agent validation passed


---
## Step 6: Chain All Agents (End-to-End)

Now run the full pipeline manually. Notice how **typed data flows** between agents:
- `TriageResult.severity` → determines routing
- `DiagnosticsResult.root_cause` → feeds remediation planning
- `RemediationPlan.action` → drives execution
- `VerificationResult.verification_status` → determines success/retry

In Challenge 2, you'll wire this into a **graph workflow** with conditional
edges — so the routing happens automatically based on these typed fields.

In [18]:
print("\U0001f3af Full Agent Pipeline Summary")
print("=" * 50)
print(f"\n1. TRIAGE: severity={triage_result.severity}, recurring={triage_result.is_recurring}")
print(f"   \u2192 Hypothesis: {triage_result.root_cause_hypothesis[:60]}...")
print(f"\n2. DIAGNOSTICS: {diag_result.root_cause[:60]}...")
print(f"   \u2192 Confidence: {diag_result.confidence}, Components: {diag_result.affected_components}")
print(f"\n3. PLAN: {plan_result.action} \u2192 {plan_result.target_details}")
print(f"   \u2192 Risk: {plan_result.risk_level}, Downtime: {plan_result.estimated_downtime_seconds}s")
print(f"\n4. VERIFY: {verify_result.verification_status}")
print(f"   \u2192 Tests: {verify_result.tests_passed} passed, {verify_result.tests_failed} failed")
print(f"\n{'='*50}")
print(f"\n\u2728 All data is TYPED \u2014 no string parsing needed.")
print(f"   This is what enables deterministic workflow routing in Challenge 2.")

🎯 Full Agent Pipeline Summary

1. TRIAGE: severity=high, recurring=True
   → Hypothesis: Memory leak in the payment-api connection pool, triggered pe...

2. DIAGNOSTICS: The payment-api pod-3 experienced a memory leak, resulting i...
   → Confidence: 0.9, Components: ['payment-api-pod-3', 'order-service']

3. PLAN: restart_pod → pod-3
   → Risk: high, Downtime: 30s

4. VERIFY: pass
   → Tests: 12 passed, 0 failed


✨ All data is TYPED — no string parsing needed.
   This is what enables deterministic workflow routing in Challenge 2.


---
## ➡️ Next: Challenge 2 — Workflow Graphs & Conditional Routing

You've been manually passing typed outputs between agents.
In Challenge 2, you'll wire these agents into a **MAF workflow graph** with:
- Switch-case routing based on `TriageResult.severity`
- State management with `ctx.set_state()`/`ctx.get_state()`
- The workflow automatically takes different paths for different incidents

[Open Challenge 2 →](../challenge-2/challenge-2.ipynb)